# 25. Reinforcement learning and exploration with bandits

**Level:** Intermediate

**Prerequisites:** 10: probability and expected value.

[Course index](00_START_HERE.ipynb) | [Previous](24_Isolation_Forest_and_Anomaly_Detection.ipynb) | [Next](26_Q_Learning_and_MDPs.ipynb)

**Topics:** Reinforcement learning, Agent, Environment, Action, Reward, Policy, Exploration, Bandits.

**Matching script:** `examples\25_Reinforcement_Learning_and_Bandits.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

A reinforcement-learning agent chooses actions, receives observations and rewards, and learns
a policy that improves expected return. Unlike a fixed supervised dataset, the agent's actions
can influence which data it receives. Define the environment, objective, action space, and
allowed interactions before selecting an algorithm.

A multi-armed bandit has action-dependent rewards without the sequential state transitions of
a full Markov decision process. Exploration tries uncertain actions; exploitation uses current
estimates. Epsilon-greedy chooses a random action with probability epsilon and otherwise
chooses the highest estimated value.

Reward is a modeling choice. Optimizing a proxy can produce unwanted behavior if the reward
omits constraints or long-term costs. A routing agent optimized only for immediate latency
might overload a link or ignore change risk. Never explore live production controls casually:
use a reviewed simulator and explicit safety limits.

## Mathematics or data contract

An incremental mean update is Q(a) <- Q(a) + (reward-Q(a))/count(a).

## Important limitation

The synthetic stationary bandit has no state, delayed consequences, or production safety constraints.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Learn which simulated action has the highest reward

**Purpose:** Implement epsilon-greedy value estimates from scratch.

### Read the code in this order
1. Define fixed reward probabilities for three safe simulated actions.
2. Explore occasionally and exploit otherwise.
3. Update only the chosen action's empirical value.
4. Check that the best estimated action matches the known simulated optimum.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
value estimates: [0.204, 0.524, 0.797]
action counts: [152, 84, 1564] mean reward: 0.734
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np

rng = np.random.default_rng(42)
reward_probabilities = np.array([0.2, 0.5, 0.8])
estimates, counts = np.zeros(3), np.zeros(3, dtype=int)
rewards = []
for _ in range(1800):
    action = int(rng.integers(3)) if rng.random() < 0.15 else int(np.argmax(estimates))
    reward = float(rng.random() < reward_probabilities[action])
    counts[action] += 1
    estimates[action] += (reward - estimates[action]) / counts[action]
    rewards.append(reward)
assert counts.sum() == 1800 and (counts > 0).all()
assert int(np.argmax(estimates)) == 2
print("value estimates:", np.round(estimates, 3).tolist())
print("action counts:", counts.tolist(), "mean reward:", round(float(np.mean(rewards)), 3))

value estimates: [0.204, 0.524, 0.797]
action counts: [152, 84, 1564] mean reward: 0.734


### Interpret example 1

The result should demonstrate this purpose: **Implement epsilon-greedy value estimates from scratch.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Worked example 2: Check discounted rewards by hand

**Purpose:** Understand why immediate reward and cumulative return are different objectives.

### Read the code in this order
1. Specify a finite reward sequence and discount factor.
2. Compute its discounted sum.
3. Compare against a delayed alternative with different gamma values.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- `raise` reports invalid input or a failed requirement instead of fabricating success.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
immediate gamma=.9: 1.0
delayed gamma=.9: 1.62
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [2]:
def discounted_return(rewards, gamma):
    if not 0 <= gamma <= 1:
        raise ValueError("gamma must be in [0, 1]")
    return sum((gamma ** index) * reward for index, reward in enumerate(rewards))
immediate, delayed = [1., 0., 0.], [0., 0., 2.]
assert discounted_return(immediate, 0.5) > discounted_return(delayed, 0.5)
assert discounted_return(immediate, 0.9) < discounted_return(delayed, 0.9)
print("immediate gamma=.9:", discounted_return(immediate, 0.9))
print("delayed gamma=.9:", round(discounted_return(delayed, 0.9), 3))

immediate gamma=.9: 1.0
delayed gamma=.9: 1.62


### Interpret example 2

The result should demonstrate this purpose: **Understand why immediate reward and cumulative return are different objectives.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Compare exploration rates on multiple seeds and report mean reward, not only one lucky run.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Reinforcement learning` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: An incremental mean update is Q(a) <- Q(a) + (reward-Q(a))/count(a).
- Name the boundary of the demonstration: The synthetic stationary bandit has no state, delayed consequences, or production safety constraints.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `http://incompleteideas.net/book/the-book-2nd.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.